In [ ]:
import os
import json
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# --- PARAMETERS ---
json_path = "WLASL_v0.3.json"
videos_folder = "videos"
frames_folder = "frames_output"
frame_rate = 15
image_size = (224, 224)
batch_size = 32
max_videos = 200
epochs = 10  # Start small to speed up training

# --- PARSE JSON AND MAP VIDEO TO LABEL ---
def parse_json(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    mapping = {}
    for entry in data:
        gloss = entry['gloss']
        for instance in entry['instances']:
            vid_id = instance['video_id']
            mapping[f"{vid_id}.mp4"] = gloss
    return mapping

video_to_label = parse_json(json_path)

# --- EXTRACT FRAMES ---
def extract_frames_from_json_map(videos_folder, output_folder, mapping, frame_rate, max_videos):
    os.makedirs(output_folder, exist_ok=True)
    count = 0
    for filename in os.listdir(videos_folder):
        if filename.endswith(".mp4") and filename in mapping:
            if count >= max_videos:
                break
            count += 1
            label = mapping[filename]
            label_folder = os.path.join(output_folder, label)
            os.makedirs(label_folder, exist_ok=True)
            cap = cv2.VideoCapture(os.path.join(videos_folder, filename))
            frame_count = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_count % frame_rate == 0:
                    resized = cv2.resize(frame, image_size)
                    out_path = os.path.join(label_folder, f"{os.path.splitext(filename)[0]}_frame{frame_count}.jpg")
                    cv2.imwrite(out_path, resized)
                frame_count += 1
            cap.release()
    print(" Frame extraction complete.")

extract_frames_from_json_map(videos_folder, frames_folder, video_to_label, frame_rate, max_videos)

# --- DATA GENERATOR ---
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    rotation_range=15
)

train_gen = datagen.flow_from_directory(
    frames_folder,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    frames_folder,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# --- MODEL ---
base_model = MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
base_model.trainable = True
for layer in base_model.layers[:100]:  # Fine-tune the last layers only
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(train_gen.num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# --- CALLBACKS ---
callbacks = [
    ReduceLROnPlateau(patience=3, factor=0.3, verbose=1),
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

# --- TRAIN MODEL ---
model.summary()
history = model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    callbacks=callbacks
)

# --- EVALUATE MODEL ---
val_loss, val_acc = model.evaluate(val_gen)
print(f"\n Final Validation Accuracy: {val_acc:.4f}")
print(f"Final Validation Loss: {val_loss:.4f}")

# --- SAVE MODEL ---
model.save('sign.keras')
print("Model saved as 'sign.keras'")


 Frame extraction complete.
Found 31477 images belonging to 12464 classes.
Found 7629 images belonging to 12464 classes.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 5,789,168 (22.08 MB)

 Trainable params: 5,392,624 (20.57 MB)

 Non-trainable params: 396,544 (1.51 MB)

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0052 - loss: 6.7114

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


984/984 ━━━━━━━━━━━━━━━━━━━━ 1805s 2s/step - accuracy: 0.0052 - loss: 6.7108 - val_accuracy: 0.0035 - val_loss: 9.9958 - learning_rate: 0.0010
Epoch 2/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 1167s 1s/step - accuracy: 0.0189 - loss: 5.4730 - val_accuracy: 0.0021 - val_loss: 13.4148 - learning_rate: 0.0010
Epoch 3/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 1370s 1s/step - accuracy: 0.0382 - loss: 4.9084 - val_accuracy: 0.0083 - val_loss: 8.0117 - learning_rate: 0.0010
Epoch 4/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 1809s 2s/step - accuracy: 0.0616 - loss: 4.5197 - val_accuracy: 0.0098 - val_loss: 7.3485 - learning_rate: 0.0010
Epoch 5/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 1350s 1s/step - accuracy: 0.0857 - loss: 4.2305 - val_accuracy: 0.0151 - val_loss: 7.5590 - learning_rate: 0.0010
Epoch 6/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 1129s 1s/step - accuracy: 0.1068 - loss: 3.9792 - val_accuracy: 0.0144 - val_loss: 10.5229 - learning_rate: 0.0010
Epoch 7/10
984/984 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1338 - loss: 3.7972


In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# --- PARAMETERS ---
frames_folder = "frames_output"
image_size = (224, 224)
batch_size = 32
initial_epoch = 8
target_epoch = 10

# --- RELOAD DATA GENERATORS ---
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    rotation_range=15
)

train_gen = datagen.flow_from_directory(
    frames_folder,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    frames_folder,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# --- LOAD MODEL FROM LAST CHECKPOINT ---
model = load_model("best_model.keras")

# --- CALLBACKS ---
callbacks = [
    ReduceLROnPlateau(patience=3, factor=0.3, verbose=1),
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True, verbose=1)
]

# --- RESUME TRAINING ---
history = model.fit(
    train_gen,
    initial_epoch=initial_epoch,
    epochs=target_epoch,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=2
)

# --- FINAL EVALUATION ---
val_loss, val_acc = model.evaluate(val_gen)
print(f"\n✅ Final Validation Accuracy: {val_acc * 100:.2f}%")
print(f"📉 Final Validation Loss: {val_loss:.4f}")

# --- SAVE FINAL MODEL ---
model.save('sign.keras')


Found 31477 images belonging to 12464 classes.
Found 7629 images belonging to 12464 classes.


C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 9/10


C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



Epoch 9: val_loss improved from inf to 6.34778, saving model to best_model.keras
984/984 - 1530s - 2s/step - accuracy: 0.2151 - loss: 3.2416 - val_accuracy: 0.0662 - val_loss: 6.3478 - learning_rate: 3.0000e-04
Epoch 10/10

Epoch 10: val_loss did not improve from 6.34778
984/984 - 1188s - 1s/step - accuracy: 0.2365 - loss: 3.1387 - val_accuracy: 0.0638 - val_loss: 6.5592 - learning_rate: 3.0000e-04
239/239 ━━━━━━━━━━━━━━━━━━━━ 189s 789ms/step - accuracy: 0.0690 - loss: 6.3426

✅ Final Validation Accuracy: 6.59%
📉 Final Validation Loss: 6.3626
